# Train LeNet on MNIST

Train the LeNet baseline. Saves the best checkpoint to `artifacts/checkpoints/lenet_mnist.pt` and metrics to `artifacts/results/training_metrics.json`.

In [ ]:
from pathlib import Path
import json
import random

import numpy as np
import onnx
import onnxruntime as ort
import torch
from torch import nn
from torch.optim import Adam
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm.auto import tqdm

In [ ]:
SEED = 42
BATCH_SIZE_TRAIN = 64
BATCH_SIZE_EVAL = 256
EPOCHS = 3
LEARNING_RATE = 1e-3

PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT / "artifacts" / "data"
CHECKPOINT_PATH = PROJECT_ROOT / "artifacts" / "checkpoints" / "lenet_mnist.pt"
ONNX_PATH = PROJECT_ROOT / "artifacts" / "onnx" / "lenet_mnist.onnx"
METRICS_PATH = PROJECT_ROOT / "artifacts" / "results" / "training_metrics.json"

for path in [DATA_ROOT, CHECKPOINT_PATH.parent, ONNX_PATH.parent, METRICS_PATH.parent]:
    path.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cpu")
print(f"Using device: {device}")

In [ ]:
class LeNetMNIST(nn.Module):
    def __init__(self, num_classes: int = 10) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 6, kernel_size=5, stride=1, padding=2),
            nn.ReLU(inplace=True),
            nn.AvgPool2d(kernel_size=2, stride=2),
            nn.Conv2d(6, 16, kernel_size=5, stride=1),
            nn.ReLU(inplace=True),
            nn.AvgPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 5 * 5, 120),
            nn.ReLU(inplace=True),
            nn.Linear(120, 84),
            nn.ReLU(inplace=True),
            nn.Linear(84, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        return self.classifier(x)

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_dataset = datasets.MNIST(root=str(DATA_ROOT), train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root=str(DATA_ROOT), train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE_TRAIN, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE_EVAL, shuffle=False, num_workers=0)

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

In [ ]:
def evaluate(model: nn.Module, loader: DataLoader, device: torch.device) -> float:
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, targets in loader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            logits = model(inputs)
            preds = logits.argmax(dim=1)
            correct += int((preds == targets).sum().item())
            total += int(targets.numel())
    return correct / total

In [ ]:
model = LeNetMNIST().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=LEARNING_RATE)

best_accuracy = 0.0
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0

    for inputs, targets in tqdm(train_loader, desc=f"Epoch {epoch}"):
        inputs = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(inputs)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)

    train_loss = running_loss / len(train_dataset)
    test_accuracy = evaluate(model, test_loader, device)
    history.append({"epoch": epoch, "train_loss": train_loss, "test_accuracy": test_accuracy})

    if test_accuracy > best_accuracy:
        best_accuracy = test_accuracy
        torch.save(model.state_dict(), CHECKPOINT_PATH)

    print(f"Epoch {epoch}: train_loss={train_loss:.4f}, test_accuracy={test_accuracy:.4f}")

print(f"Best checkpoint saved to: {CHECKPOINT_PATH}")
print(f"Best test accuracy: {best_accuracy:.4f}")

In [ ]:
metrics_payload = {
    "seed": SEED,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "best_test_accuracy": best_accuracy,
    "checkpoint_path": str(CHECKPOINT_PATH.relative_to(PROJECT_ROOT)),
    "history": history,
}

with METRICS_PATH.open("w", encoding="utf-8") as handle:
    json.dump(metrics_payload, handle, indent=2)

metrics_payload

## Export to ONNX

Reload the best checkpoint and export to ONNX (opset 18, dynamic batch axis).

In [ ]:
export_model = LeNetMNIST().to(device)
export_model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
export_model.eval()

dummy_input = torch.randn(1, 1, 28, 28, device=device)

torch.onnx.export(
    export_model,
    (dummy_input,),
    str(ONNX_PATH),
    input_names=["input"],
    output_names=["logits"],
    dynamic_axes={"input": {0: "batch_size"}, "logits": {0: "batch_size"}},
    opset_version=18,
    dynamo=True,
)

onnx_model = onnx.load(str(ONNX_PATH))
onnx.checker.check_model(onnx_model)

print(f"Saved ONNX model to: {ONNX_PATH}")
print(f"ONNX graph nodes: {len(onnx_model.graph.node)}")

## ONNX vs PyTorch sanity check

Quick check that ONNX Runtime and PyTorch agree on the test set before going on to TVM.

In [ ]:
ort_session = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])
ort_input_name = ort_session.get_inputs()[0].name

total_samples = 0
pytorch_correct = 0
onnx_correct = 0
matching_predictions = 0
max_abs_diff = 0.0
mismatch_examples = []

for batch_index, (inputs, targets) in enumerate(test_loader):
    with torch.no_grad():
        pytorch_logits = model(inputs.to(device)).cpu().numpy()

    onnx_logits = ort_session.run(None, {ort_input_name: inputs.numpy().astype(np.float32)})[0]
    targets_np = targets.numpy()

    pytorch_preds = np.argmax(pytorch_logits, axis=1)
    onnx_preds = np.argmax(onnx_logits, axis=1)

    pytorch_correct += int((pytorch_preds == targets_np).sum())
    onnx_correct += int((onnx_preds == targets_np).sum())
    matching_predictions += int((pytorch_preds == onnx_preds).sum())
    total_samples += int(targets_np.shape[0])

    batch_max_diff = float(np.max(np.abs(pytorch_logits - onnx_logits)))
    max_abs_diff = max(max_abs_diff, batch_max_diff)

    if len(mismatch_examples) < 10:
        mismatch_indices = np.where(pytorch_preds != onnx_preds)[0]
        for idx in mismatch_indices:
            mismatch_examples.append({
                "batch_index": int(batch_index),
                "sample_in_batch": int(idx),
                "target": int(targets_np[idx]),
                "pytorch_pred": int(pytorch_preds[idx]),
                "onnx_pred": int(onnx_preds[idx]),
            })
            if len(mismatch_examples) >= 10:
                break

verification_results = {
    "num_samples": total_samples,
    "pytorch_accuracy": pytorch_correct / total_samples,
    "onnx_accuracy": onnx_correct / total_samples,
    "prediction_match_rate": matching_predictions / total_samples,
    "max_abs_logit_difference": max_abs_diff,
    "num_prediction_mismatches": total_samples - matching_predictions,
    "mismatch_examples": mismatch_examples,
}

metrics_payload["onnx_verification"] = verification_results

with METRICS_PATH.open("w", encoding="utf-8") as handle:
    json.dump(metrics_payload, handle, indent=2)

verification_results

In [ ]:
print(f"PyTorch accuracy: {verification_results['pytorch_accuracy']:.4f}")
print(f"ONNX accuracy: {verification_results['onnx_accuracy']:.4f}")
print(f"Prediction match rate: {verification_results['prediction_match_rate']:.4f}")
print(f"Max absolute logit difference: {verification_results['max_abs_logit_difference']:.6f}")
print(f"Prediction mismatches: {verification_results['num_prediction_mismatches']}")
print(f"Updated metrics file: {METRICS_PATH}")